# LandingAI Agentic Document Extraction (ADE) - All Case Studies

This notebook evaluates **LandingAI ADE** across five challenging document types to benchmark its performance against LLMWhisperer and Pytesseract.

## Document Cases:
1. **Tax Form (ACORD Insurance Application)** - Complex multi-column form with checkboxes
2. **Hotel Receipt (Photographed)** - Real-world photograph with lighting challenges
3. **German Insurance Form (Multilingual)** - German-English bilingual document with special characters
4. **Misaligned Packing List (Scanned)** - Skewed logistics document with dense tables
5. **Handwritten Scanned Document** - Insurance form with handwritten entries

## LandingAI ADE Features:
- Document Pre-Trained Transformer (DPT-2) model
- Layout-preserving parsing
- Structured markdown output with chunk metadata
- Grounding information for each extracted element

In [1]:
# Import required libraries
import os
import time
import re
from pathlib import Path
from dotenv import load_dotenv
from landingai_ade import LandingAIADE

# Load environment variables
load_dotenv(override=True)

# Verify API key
API_KEY = os.getenv("VISION_AGENT_API_KEY")
if not API_KEY:
    raise ValueError("VISION_AGENT_API_KEY not found in environment variables")

# Initialize the LandingAI ADE client
client = LandingAIADE()

print("✓ LandingAI ADE client initialized successfully!")

✓ LandingAI ADE client initialized successfully!


In [2]:
# Define document paths for all 5 cases
documents = {
    "tax_form": {
        "path": "docs/image-ocr-insurance-form (1).png",
        "name": "Tax Form (ACORD Insurance Application)",
        "challenges": ["Multi-column layout", "Checkboxes", "Dense text", "Tables"]
    },
    "hotel_receipt": {
        "path": "docs/receipt-hotel-photograph.png",
        "name": "Hotel Receipt (Photographed)",
        "challenges": ["Photograph quality", "Uneven lighting", "Handwritten elements", "Mixed fonts"]
    },
    "german_form": {
        "path": "docs/Insurance-form-german.pdf",
        "name": "German Insurance Form (Multilingual)",
        "challenges": ["German umlauts (ä, ö, ü)", "Bilingual labels", "Special characters", "€ symbol"]
    },
    "packing_list": {
        "path": "docs/scanned-misaligned-logistics_packing_list (1).pdf",
        "name": "Misaligned Packing List (Scanned)",
        "challenges": ["Page skew", "Dense tables", "Multi-column sections", "Fine print"]
    },
    "handwritten": {
        "path": "docs/accord-insurance-handwritten-scanned-copy.pdf",
        "name": "Handwritten Scanned Insurance Form",
        "challenges": ["Handwritten text", "Scan quality", "Mixed print/handwriting", "Form fields"]
    }
}

# Verify all files exist
print("Document Inventory:")
print("=" * 80)
for doc_id, doc_info in documents.items():
    path = Path(doc_info["path"])
    exists = path.exists()
    size = path.stat().st_size / 1024 if exists else 0
    status = "✓" if exists else "✗"
    print(f"{status} {doc_info['name']}")
    print(f"  Path: {doc_info['path']}")
    print(f"  Size: {size:.2f} KB")
    print(f"  Challenges: {', '.join(doc_info['challenges'])}")
    print()

Document Inventory:
✓ Tax Form (ACORD Insurance Application)
  Path: docs/image-ocr-insurance-form (1).png
  Size: 442.77 KB
  Challenges: Multi-column layout, Checkboxes, Dense text, Tables

✓ Hotel Receipt (Photographed)
  Path: docs/receipt-hotel-photograph.png
  Size: 9416.39 KB
  Challenges: Photograph quality, Uneven lighting, Handwritten elements, Mixed fonts

✓ German Insurance Form (Multilingual)
  Path: docs/Insurance-form-german.pdf
  Size: 90.04 KB
  Challenges: German umlauts (ä, ö, ü), Bilingual labels, Special characters, € symbol

✓ Misaligned Packing List (Scanned)
  Path: docs/scanned-misaligned-logistics_packing_list (1).pdf
  Size: 764.98 KB
  Challenges: Page skew, Dense tables, Multi-column sections, Fine print

✓ Handwritten Scanned Insurance Form
  Path: docs/accord-insurance-handwritten-scanned-copy.pdf
  Size: 579.29 KB
  Challenges: Handwritten text, Scan quality, Mixed print/handwriting, Form fields



In [3]:
# Helper function to extract text with LandingAI ADE
def extract_with_landingai(file_path: str, doc_name: str) -> dict:
    """Extract text from document using LandingAI ADE.
    
    Args:
        file_path: Path to the document file.
        doc_name: Human-readable document name.
        
    Returns:
        Dictionary with extraction results and metadata.
    """
    print(f"{'='*80}")
    print(f"LANDINGAI ADE EXTRACTION: {doc_name}")
    print(f"{'='*80}")
    print(f"Processing: {file_path}")
    
    start_time = time.time()
    
    try:
        response = client.parse(
            document=Path(file_path),
            model="dpt-2-latest"
        )
        
        processing_time = time.time() - start_time
        
        # Extract key metrics
        markdown_text = response.markdown
        chunks = response.chunks
        metadata = response.metadata
        
        print(f"\n✓ Extraction complete")
        print(f"  Characters: {len(markdown_text):,}")
        print(f"  Lines: {len(markdown_text.splitlines())}")
        print(f"  Chunks: {len(chunks)}")
        print(f"  Processing time: {processing_time:.2f} seconds")
        
        if hasattr(metadata, 'page_count'):
            print(f"  Pages: {metadata.page_count}")
        if hasattr(metadata, 'credit_usage'):
            print(f"  Credits used: {metadata.credit_usage}")
        
        return {
            "success": True,
            "markdown": markdown_text,
            "chunks": chunks,
            "metadata": metadata,
            "processing_time": processing_time,
            "char_count": len(markdown_text),
            "line_count": len(markdown_text.splitlines()),
            "chunk_count": len(chunks)
        }
        
    except Exception as e:
        processing_time = time.time() - start_time
        print(f"\n✗ Extraction failed: {str(e)}")
        return {
            "success": False,
            "error": str(e),
            "processing_time": processing_time
        }

print("✓ Extraction function defined")

✓ Extraction function defined


---
## Case 1: Tax Form (ACORD Commercial Insurance Application)

Complex multi-column form with checkboxes, tables, and dense text layouts.

In [4]:
# Extract Tax Form
tax_form_result = extract_with_landingai(
    documents["tax_form"]["path"],
    documents["tax_form"]["name"]
)

if tax_form_result["success"]:
    print(f"\n{'='*80}")
    print("PREVIEW: First 1500 characters")
    print(f"{'='*80}")
    print(tax_form_result["markdown"][:1500])
    print("\n... (truncated)")

LANDINGAI ADE EXTRACTION: Tax Form (ACORD Insurance Application)
Processing: docs/image-ocr-insurance-form (1).png

✓ Extraction complete
  Characters: 9,904
  Lines: 129
  Chunks: 12
  Processing time: 9.43 seconds
  Pages: 1
  Credits used: 3.0

PREVIEW: First 1500 characters
<a id='cdb32624-6bcb-4a8f-a4f0-0b728446c893'></a>

<::logo: ACORD
ACORD
Black text with a curved line underneath, forming a stylized "ACORD" logo.::>

<a id='69bf6282-61f8-408d-90c0-8912036cbace'></a>

COMMERCIAL INSURANCE APPLICATION

APPLICANT INFORMATION SECTION

<a id='adb0363b-fd69-4711-84c6-001ad04caa23'></a>

DATE (MM/DD/YYYY)
08/20/2023

<a id='325c0cd8-ba20-4e43-87bd-7cab21eaed23'></a>

<table id="0-1">
<tr><td id="0-2" colspan="2">AGENCY Smith Insurance Agency 123 Main Street, Anytown, NY, 10001</td></tr>
<tr><td id="0-3" colspan="2">CONTACT NAME: John Smith</td></tr>
<tr><td id="0-4" colspan="2">PHONE (A/C. No. Ext): (555) 123-4567</td></tr>
<tr><td id="0-5" colspan="2">FAX (A/C. No): (555) 123-4567</

In [5]:
# Analyze Tax Form extraction quality
if tax_form_result["success"]:
    text = tax_form_result["markdown"]
    
    print("="*80)
    print("TAX FORM QUALITY ANALYSIS")
    print("="*80)
    
    # Key fields to detect
    key_fields = [
        "AGENCY", "CARRIER", "POLICY NUMBER", "SECTIONS ATTACHED",
        "PREMIUM", "APPLICANT INFORMATION", "UNDERWRITER", "NAIC CODE"
    ]
    
    print("\n📋 Key Field Detection:")
    found_count = 0
    for field in key_fields:
        found = field.lower() in text.lower()
        status = "✓" if found else "✗"
        print(f"  {status} {field}")
        if found:
            found_count += 1
    
    print(f"\n  Detection Rate: {found_count}/{len(key_fields)} ({found_count/len(key_fields)*100:.1f}%)")
    
    # Checkbox detection
    checkbox_patterns = re.findall(r'[\[\(][xX✓✗ ][\]\)]|☑|☐|⊠', text)
    print(f"\n📦 Checkbox Detection: {len(checkbox_patterns)} marks found")
    
    # Premium amounts
    amounts = re.findall(r'\$\s*[\d,]+(?:\.\d{2})?', text)
    print(f"\n💰 Premium Amounts Found: {len(amounts)}")
    if amounts:
        print(f"    Examples: {', '.join(amounts[:5])}")

TAX FORM QUALITY ANALYSIS

📋 Key Field Detection:
  ✓ AGENCY
  ✓ CARRIER
  ✓ POLICY NUMBER
  ✓ SECTIONS ATTACHED
  ✓ PREMIUM
  ✓ APPLICANT INFORMATION
  ✓ UNDERWRITER
  ✓ NAIC CODE

  Detection Rate: 8/8 (100.0%)

📦 Checkbox Detection: 0 marks found

💰 Premium Amounts Found: 9
    Examples: $ 873.05, $ 364.03, $ 9.04, $ 391.00, $ 495.02


In [6]:
# Save Tax Form output
if tax_form_result["success"]:
    with open("extracted_text_tax_form_landingai.txt", 'w', encoding='utf-8') as f:
        f.write(tax_form_result["markdown"])
    print("✓ Saved to: extracted_text_tax_form_landingai.txt")

✓ Saved to: extracted_text_tax_form_landingai.txt


---
## Case 2: Hotel Receipt (Photographed)

Real-world photograph with lighting challenges, shadows, and handwritten elements.

In [7]:
# Extract Hotel Receipt
receipt_result = extract_with_landingai(
    documents["hotel_receipt"]["path"],
    documents["hotel_receipt"]["name"]
)

if receipt_result["success"]:
    print(f"\n{'='*80}")
    print("PREVIEW: First 1500 characters")
    print(f"{'='*80}")
    print(receipt_result["markdown"][:1500])
    print("\n... (truncated)")

LANDINGAI ADE EXTRACTION: Hotel Receipt (Photographed)
Processing: docs/receipt-hotel-photograph.png

✓ Extraction complete
  Characters: 2,875
  Lines: 104
  Chunks: 15
  Processing time: 18.36 seconds
  Pages: 1
  Credits used: 3.0

PREVIEW: First 1500 characters
<a id='eaddd9e3-f069-4dcc-a71f-b701fec3b528'></a>

INTERCONTINENTAL.

CHENNAI MAHABALIPURAM RESORT
No.212, East Coast Road, Nemelli Village, Perur Post Office, Chengalpattu District
Chennai - 603 104. Tamil Nadu, India.

Tel : +91 44 7172 0101 Fax: +91 44 7172 0102 intercontinental.com/Chennai

GSTIN No.: 33AAACA9041L1ZR
Serial No 367005 TAX INVOICE: 33AAACA9041L1ZR

<a id='098c87bf-6005-4b00-9cc6-9519c524e2c8'></a>

<::logo: Tao of Peng
tao of peng
Journey of a thousand flavours
The logo features a silhouette of a Chinese hat or a wok-like shape in black and white, enclosing the brand name.::>

<a id='e84334aa-77d1-4943-99e9-62b263001571'></a>

<::logo: The Melting Pot
the melting pot
market cafe
A black and white logo feat

In [8]:
# Analyze Hotel Receipt extraction quality
if receipt_result["success"]:
    text = receipt_result["markdown"]
    
    print("="*80)
    print("HOTEL RECEIPT QUALITY ANALYSIS")
    print("="*80)
    
    # Key fields to detect
    key_fields = [
        "INTERCONTINENTAL", "TAX INVOICE", "Guest Name", "Room No",
        "Subtotal", "Service Charge", "GSTIN", "Total"
    ]
    
    print("\n📋 Key Field Detection:")
    found_count = 0
    for field in key_fields:
        found = field.lower() in text.lower()
        status = "✓" if found else "✗"
        print(f"  {status} {field}")
        if found:
            found_count += 1
    
    print(f"\n  Detection Rate: {found_count}/{len(key_fields)} ({found_count/len(key_fields)*100:.1f}%)")
    
    # Amount extraction (Rs amounts)
    amounts = re.findall(r'Rs\.?\s*[\d,]+(?:\.\d+)?', text)
    print(f"\n💰 Amounts Found: {len(amounts)}")
    if amounts:
        print(f"    Examples: {', '.join(amounts[:5])}")
    
    # Guest name detection
    if "simon" in text.lower() or "dawes" in text.lower():
        print("\n✓ Guest name (Simon Dawes) detected")
    else:
        print("\n✗ Guest name not clearly detected")

HOTEL RECEIPT QUALITY ANALYSIS

📋 Key Field Detection:
  ✓ INTERCONTINENTAL
  ✓ TAX INVOICE
  ✓ Guest Name
  ✓ Room No
  ✓ Subtotal
  ✓ Service Charge
  ✓ GSTIN
  ✓ Total

  Detection Rate: 8/8 (100.0%)

💰 Amounts Found: 5
    Examples: Rs750.00, Rs37.50, Rs70.88, Rs70.88, Rs929.26

✓ Guest name (Simon Dawes) detected


In [ ]:
# Save Hotel Receipt output
if receipt_result["success"]:
    with open("extracted_text_receipt_landingai.txt", 'w', encoding='utf-8') as f:
        f.write(receipt_result["markdown"])
    print("✓ Saved to: extracted_text_receipt_landingai.txt")

---
## Case 3: German Insurance Form (Multilingual)

Bilingual German-English document with umlauts and special characters.

In [9]:
# Extract German Form
german_result = extract_with_landingai(
    documents["german_form"]["path"],
    documents["german_form"]["name"]
)

if german_result["success"]:
    print(f"\n{'='*80}")
    print("PREVIEW: First 1500 characters")
    print(f"{'='*80}")
    print(german_result["markdown"][:1500])
    print("\n... (truncated)")

LANDINGAI ADE EXTRACTION: German Insurance Form (Multilingual)
Processing: docs/Insurance-form-german.pdf

✓ Extraction complete
  Characters: 3,939
  Lines: 188
  Chunks: 9
  Processing time: 7.21 seconds
  Pages: 1
  Credits used: 3.0

PREVIEW: First 1500 characters
<a id='ffd02c3e-e0b6-4f7c-855d-fd6c0906593c'></a>

In welcher Währung möchten Sie Ihre Prämie zahlen? Ihre Versicherungsleistungen werden ebenfalls in dieser Währung erfolgen.
In which currency would you like to pay your premium? Your policy benefits will also be in this currency.
option GB£: [ ]
option Euro€: [x]
option US$: [ ]

Wie viel Selbstbeteiligung möchten Sie übernehmen? Selbstbeteiligung gilt pro Person pro Versicherungsjahr und nicht für die Optionen Reguläre Schwangerschaft und Entbindung, Zahnärztliche Behandlung, Evakuierung oder Repatriierung oder Leistungen für Wellness, Optik und Impfungen. Wählen Sie eine höhere Selbstbeteiligung, um Ihre Versicherungsprämie zu reduzieren.
How much excess would you like

In [10]:
# Analyze German Form extraction quality
if german_result["success"]:
    text = german_result["markdown"]
    
    print("="*80)
    print("GERMAN FORM QUALITY ANALYSIS")
    print("="*80)
    
    # German special characters
    german_chars = ['ä', 'ö', 'ü', 'Ä', 'Ö', 'Ü', 'ß', '€']
    
    print("\n🔤 German Special Character Detection:")
    total_special = 0
    for char in german_chars:
        count = text.count(char)
        total_special += count
        if count > 0:
            print(f"  {char}: {count} occurrences")
    
    print(f"\n  Total special characters: {total_special}")
    
    # Bilingual field detection
    german_words = ['versicherung', 'antrag', 'geburtsdatum', 'adresse', 'prämie']
    english_words = ['insurance', 'application', 'date of birth', 'address', 'premium']
    
    german_found = sum(1 for w in german_words if w in text.lower())
    english_found = sum(1 for w in english_words if w in text.lower())
    
    print(f"\n🌍 Language Detection:")
    print(f"  German words found: {german_found}/{len(german_words)}")
    print(f"  English words found: {english_found}/{len(english_words)}")
    print(f"  Bilingual: {'✓ Yes' if german_found > 0 and english_found > 0 else '✗ No'}")
    
    # Currency detection
    euro_amounts = re.findall(r'€\s*[\d.,]+', text)
    print(f"\n💶 Euro Amounts Found: {len(euro_amounts)}")
    if euro_amounts:
        print(f"    Examples: {', '.join(euro_amounts[:5])}")

GERMAN FORM QUALITY ANALYSIS

🔤 German Special Character Detection:
  ä: 12 occurrences
  ö: 5 occurrences
  ü: 7 occurrences
  €: 9 occurrences

  Total special characters: 33

🌍 Language Detection:
  German words found: 5/5
  English words found: 5/5
  Bilingual: ✓ Yes

💶 Euro Amounts Found: 8
    Examples: € 60, € 180, € 360, € 600, € 1,200


In [ ]:
# Save German Form output
if german_result["success"]:
    with open("extracted_text_german_form_landingai.txt", 'w', encoding='utf-8') as f:
        f.write(german_result["markdown"])
    print("✓ Saved to: extracted_text_german_form_landingai.txt")

---
## Case 4: Misaligned Packing List (Scanned)

Skewed logistics document with dense tabular data and multi-column sections.

In [11]:
# Extract Packing List
packing_result = extract_with_landingai(
    documents["packing_list"]["path"],
    documents["packing_list"]["name"]
)

if packing_result["success"]:
    print(f"\n{'='*80}")
    print("PREVIEW: First 1500 characters")
    print(f"{'='*80}")
    print(packing_result["markdown"][:1500])
    print("\n... (truncated)")

LANDINGAI ADE EXTRACTION: Misaligned Packing List (Scanned)
Processing: docs/scanned-misaligned-logistics_packing_list (1).pdf

✓ Extraction complete
  Characters: 2,381
  Lines: 20
  Chunks: 1
  Processing time: 11.79 seconds
  Pages: 1
  Credits used: 3.0

PREVIEW: First 1500 characters
<a id='9aeb816e-0799-4b4a-a388-7727e761018f'></a>

<table id="0-1">
<tr><td id="0-2" colspan="4">Packing List</td></tr>
<tr><td id="0-3">Shipper/ Exporter: Faculty of Arts 5 Washington Square S, New York, NY 10012, USA</td><td id="0-4">Ultimate Consignee: Herald Corp 28 OLD BROMPTON ROAD, SOUTH KENSINGTON</td><td id="0-5">Faculty of Arts 5 Washington Square S, New York, NY 10012, USA</td><td id="0-6">Intermediate Consignee</td></tr>
<tr><td id="0-7" rowspan="2">Commercial Invoice No.:894933 Order No.: AWB/BL Number: Date Of Shipment: Currency: 34343454 Freight: 12/12/2023 Dollars</td><td id="0-8" rowspan="2">Total number of Packages: 32 Total Gross Weight (Lbs): Total Gross Weight (Kgs): 35 Total Net 

In [12]:
# Analyze Packing List extraction quality
if packing_result["success"]:
    text = packing_result["markdown"]
    
    print("="*80)
    print("PACKING LIST QUALITY ANALYSIS")
    print("="*80)
    
    # Key sections to detect
    key_sections = [
        "SHIPPER", "CONSIGNEE", "BILL TO", "PACKING LIST", "INVOICE",
        "DESCRIPTION", "QUANTITY", "WEIGHT", "DIMENSIONS", "SIGNATURE", "DATE", "TOTAL"
    ]
    
    print("\n📋 Section Detection:")
    found_count = 0
    for section in key_sections:
        found = section.lower() in text.lower()
        status = "✓" if found else "✗"
        print(f"  {status} {section}")
        if found:
            found_count += 1
    
    print(f"\n  Detection Rate: {found_count}/{len(key_sections)} ({found_count/len(key_sections)*100:.1f}%)")
    
    # Table structure indicators
    has_table_markdown = '|' in text and '-' in text
    print(f"\n📊 Table Structure:")
    print(f"  Markdown table detected: {'✓ Yes' if has_table_markdown else '✗ No'}")
    
    # Address/ZIP detection
    zips = re.findall(r'\b\d{5}(?:-\d{4})?\b', text)
    print(f"\n📮 ZIP Codes Found: {len(zips)}")
    if zips:
        print(f"    Examples: {', '.join(zips[:5])}")

PACKING LIST QUALITY ANALYSIS

📋 Section Detection:
  ✓ SHIPPER
  ✓ CONSIGNEE
  ✗ BILL TO
  ✓ PACKING LIST
  ✓ INVOICE
  ✓ DESCRIPTION
  ✓ QUANTITY
  ✓ WEIGHT
  ✓ DIMENSIONS
  ✓ SIGNATURE
  ✓ DATE
  ✓ TOTAL

  Detection Rate: 11/12 (91.7%)

📊 Table Structure:
  Markdown table detected: ✗ No

📮 ZIP Codes Found: 3
    Examples: 10012, 10012, 23445


In [ ]:
# Save Packing List output
if packing_result["success"]:
    with open("extracted_text_packing_list_landingai.txt", 'w', encoding='utf-8') as f:
        f.write(packing_result["markdown"])
    print("✓ Saved to: extracted_text_packing_list_landingai.txt")

---
## Case 5: Handwritten Scanned Document

Insurance form with handwritten entries, challenging OCR scenario.

In [13]:
# Extract Handwritten Document
handwritten_result = extract_with_landingai(
    documents["handwritten"]["path"],
    documents["handwritten"]["name"]
)

if handwritten_result["success"]:
    print(f"\n{'='*80}")
    print("PREVIEW: First 1500 characters")
    print(f"{'='*80}")
    print(handwritten_result["markdown"][:1500])
    print("\n... (truncated)")

LANDINGAI ADE EXTRACTION: Handwritten Scanned Insurance Form
Processing: docs/accord-insurance-handwritten-scanned-copy.pdf

✓ Extraction complete
  Characters: 8,949
  Lines: 123
  Chunks: 12
  Processing time: 9.93 seconds
  Pages: 1
  Credits used: 3.0

PREVIEW: First 1500 characters
<a id='da66e3c6-30dd-4843-a9c3-2738bab23ec8'></a>

<:: ACORD® logo : figure::>

<a id='abce866b-4dbf-4644-809e-78b16356cfbc'></a>

COMMERCIAL INSURANCE APPLICATION
APPLICANT INFORMATION SECTION

<a id='9dba9526-75f9-4aaf-ac38-7da734347fc4'></a>

DATE (MM/DD/YYYY)
03/02/2024

<a id='79f3c994-6ed4-4c48-865b-e27e25db828b'></a>

<table id="0-1">
<tr><td id="0-2" colspan="2">AGENCY Fincorp insurance</td></tr>
<tr><td id="0-3" colspan="2">CONTACT NAME: Simon</td></tr>
<tr><td id="0-4" colspan="2">PHONE (A/C, No, Ext): 23986582</td></tr>
<tr><td id="0-5" colspan="2">FAX (A/C, No):</td></tr>
<tr><td id="0-6" colspan="2">E-MAIL ADDRESS:</td></tr>
<tr><td id="0-7">CODE:</td><td id="0-8">SUBCODE:</td></tr>
<tr><td

In [14]:
# Analyze Handwritten Document extraction quality
if handwritten_result["success"]:
    text = handwritten_result["markdown"]
    
    print("="*80)
    print("HANDWRITTEN DOCUMENT QUALITY ANALYSIS")
    print("="*80)
    
    # Insurance form key fields
    key_fields = [
        "AGENCY", "CARRIER", "POLICY", "APPLICANT", "INSURANCE",
        "NAME", "ADDRESS", "DATE", "SIGNATURE"
    ]
    
    print("\n📋 Key Field Detection:")
    found_count = 0
    for field in key_fields:
        found = field.lower() in text.lower()
        status = "✓" if found else "✗"
        print(f"  {status} {field}")
        if found:
            found_count += 1
    
    print(f"\n  Detection Rate: {found_count}/{len(key_fields)} ({found_count/len(key_fields)*100:.1f}%)")
    
    # Character analysis (handwriting tends to have more errors)
    words = text.split()
    print(f"\n📝 Text Statistics:")
    print(f"  Total words: {len(words)}")
    print(f"  Total characters: {len(text):,}")
    print(f"  Total lines: {len(text.splitlines())}")

HANDWRITTEN DOCUMENT QUALITY ANALYSIS

📋 Key Field Detection:
  ✓ AGENCY
  ✓ CARRIER
  ✓ POLICY
  ✓ APPLICANT
  ✓ INSURANCE
  ✓ NAME
  ✓ ADDRESS
  ✓ DATE
  ✓ SIGNATURE

  Detection Rate: 9/9 (100.0%)

📝 Text Statistics:
  Total words: 699
  Total characters: 8,949
  Total lines: 123


In [ ]:
# Save Handwritten Document output
if handwritten_result["success"]:
    with open("extracted_text_handwritten_landingai.txt", 'w', encoding='utf-8') as f:
        f.write(handwritten_result["markdown"])
    print("✓ Saved to: extracted_text_handwritten_landingai.txt")

---
## Comprehensive Comparison Summary

In [15]:
# Compile all results
all_results = {
    "Tax Form": tax_form_result,
    "Hotel Receipt": receipt_result,
    "German Form": german_result,
    "Packing List": packing_result,
    "Handwritten": handwritten_result
}

print("="*100)
print("LANDINGAI ADE PERFORMANCE SUMMARY")
print("="*100)
print()
print(f"{'Document':<25} {'Status':<12} {'Characters':<15} {'Lines':<10} {'Chunks':<10} {'Time (s)':<10}")
print("-" * 100)

total_chars = 0
total_time = 0
success_count = 0

for doc_name, result in all_results.items():
    if result["success"]:
        chars = result["char_count"]
        lines = result["line_count"]
        chunks = result["chunk_count"]
        time_taken = result["processing_time"]
        
        total_chars += chars
        total_time += time_taken
        success_count += 1
        
        print(f"{doc_name:<25} {'✓ Success':<12} {chars:,}{'':>5} {lines:<10} {chunks:<10} {time_taken:.2f}")
    else:
        print(f"{doc_name:<25} {'✗ Failed':<12} {'-':<15} {'-':<10} {'-':<10} {result['processing_time']:.2f}")
        total_time += result['processing_time']

print("-" * 100)
print(f"{'TOTALS':<25} {f'{success_count}/5':<12} {total_chars:,}{'':>5} {'':<10} {'':<10} {total_time:.2f}")
print()
print(f"Success Rate: {success_count}/5 ({success_count/5*100:.0f}%)")
print(f"Average Processing Time: {total_time/5:.2f} seconds per document")

LANDINGAI ADE PERFORMANCE SUMMARY

Document                  Status       Characters      Lines      Chunks     Time (s)  
----------------------------------------------------------------------------------------------------
Tax Form                  ✓ Success    9,904      129        12         9.43
Hotel Receipt             ✓ Success    2,875      104        15         18.36
German Form               ✓ Success    3,939      188        9          7.21
Packing List              ✓ Success    2,381      20         1          11.79
Handwritten               ✓ Success    8,949      123        12         9.93
----------------------------------------------------------------------------------------------------
TOTALS                    5/5          28,048                            56.72

Success Rate: 5/5 (100%)
Average Processing Time: 11.34 seconds per document


In [16]:
# Compare with LLMWhisperer benchmarks (from previous notebooks)
print("\n" + "="*100)
print("COMPARISON WITH LLMWHISPERER BENCHMARKS")
print("="*100)

# LLMWhisperer results from previous analysis
llmwhisperer_benchmarks = {
    "Tax Form": {"chars": 7066, "time": 11.31},
    "Hotel Receipt": {"chars": 2314, "time": 36.30},
    "German Form": {"chars": 6691, "time": 11.14},
    "Packing List": {"chars": 2547, "time": 11.70},
    "Handwritten": {"chars": 0, "time": 0}  # Placeholder - need actual data
}

print()
print(f"{'Document':<25} {'LandingAI Chars':<18} {'LLMWhisperer Chars':<20} {'LandingAI Time':<16} {'LLMWhisperer Time'}")
print("-" * 100)

for doc_name, result in all_results.items():
    llm_data = llmwhisperer_benchmarks.get(doc_name, {"chars": 0, "time": 0})
    
    if result["success"]:
        landing_chars = result["char_count"]
        landing_time = result["processing_time"]
    else:
        landing_chars = 0
        landing_time = 0
    
    llm_chars = llm_data["chars"]
    llm_time = llm_data["time"]
    
    print(f"{doc_name:<25} {landing_chars:,}{'':>8} {llm_chars:,}{'':>10} {landing_time:.2f}s{'':>8} {llm_time:.2f}s")

print("-" * 100)


COMPARISON WITH LLMWHISPERER BENCHMARKS

Document                  LandingAI Chars    LLMWhisperer Chars   LandingAI Time   LLMWhisperer Time
----------------------------------------------------------------------------------------------------
Tax Form                  9,904         7,066           9.43s         11.31s
Hotel Receipt             2,875         2,314           18.36s         36.30s
German Form               3,939         6,691           7.21s         11.14s
Packing List              2,381         2,547           11.79s         11.70s
Handwritten               8,949         0           9.93s         0.00s
----------------------------------------------------------------------------------------------------


In [ ]:
# Final conclusions
print("\n" + "="*100)
print("CONCLUSIONS")
print("="*100)

print("""
🎯 LANDINGAI ADE PERFORMANCE SUMMARY:

Strengths:
  • Modern Document Pre-Trained Transformer (DPT-2) model
  • Structured markdown output with semantic chunks
  • Grounding information for each extracted element
  • Built-in support for complex document layouts
  • API-first design with comprehensive Python library

Best Use Cases:
  • Documents requiring structured data extraction
  • Workflows needing element-level grounding/coordinates
  • Integration with downstream LLM pipelines
  • Applications requiring split and extract functionality

Comparison Notes:
  • Both LandingAI ADE and LLMWhisperer significantly outperform traditional OCR
  • Processing times and character counts may vary based on document complexity
  • LandingAI provides native markdown with chunk metadata
  • LLMWhisperer provides layout-preserving plain text output

Recommendation:
  Choose based on your specific workflow requirements:
  - LandingAI ADE: When you need structured extraction with schemas
  - LLMWhisperer: When you need layout-preserving text for LLM context
  - Pytesseract: Only for simple, clean documents where speed is critical
""")